# TrustDoc — GPU + Toolchain Verification

Run this top-to-bottom on **Kaggle** (enable GPU T4x2/P100 accelerator) or **Colab** (Runtime > Change runtime type > GPU). It checks:

1. GPU is actually visible to PyTorch
2. Whether the environment's `transformers` version has a real LayoutLMv3 slowdown regression — checked empirically (timed benchmark), not by asserting an exact version string (an old exact pin fights Kaggle's current Python 3.12 image; see [issue #30974](https://github.com/huggingface/transformers/issues/30974) for the historical bug this guards against)
3. A real LayoutLMv3 forward+backward pass on FUNSD runs at a sane steps/sec

If step 3 looks off by an order of magnitude, stop and investigate before starting any real training.

**If you hit a broken transformers import** (e.g. `cannot import name 'is_flax_available'`), a prior pip install partially corrupted the on-disk package — restart the session fully (fresh container) before re-running; no cell in this notebook can repair a half-overwritten install.

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU visible — on Kaggle: Settings > Accelerator > GPU T4x2/P100. On Colab: Runtime > Change runtime type > GPU.")

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        print(f"{pkg}: already available, skipping install")
    except ImportError:
        # --no-deps: avoid letting pip's resolver touch the existing transformers/torch/huggingface_hub stack
        print(f"{pkg}: not found, installing with --no-deps")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg], check=True)

# only installing what this notebook's code actually uses: load_dataset (datasets) and the raw
# torch training loop below. No seqeval (no F1 scoring here) or accelerate (no Trainer here).
ensure("datasets")

import transformers
print("transformers version (as provided by this environment):", transformers.__version__)
print("Not touching the transformers install at all here — Kaggle already ships a working, current version. "
      "The real regression check is the timed benchmark below: if step time comes back sane, there is no "
      "4.34.0-era regression (issue #30974) in play, regardless of the exact version string.")


In [ ]:
from datasets import load_dataset
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification

dataset = load_dataset("nielsr/funsd-layoutlmv3")
print("dataset columns:", dataset["train"].column_names)
label_list = dataset["train"].features["ner_tags"].feature.names
print("labels:", label_list)

# HF NER-style datasets conventionally call the word list "tokens", not "words" -- detect rather than assume
text_column = next(c for c in ("tokens", "words") if c in dataset["train"].column_names)
print("using text column:", text_column)

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

def prepare(examples):
    return processor(
        examples["image"],
        examples[text_column],
        boxes=examples["bboxes"],
        word_labels=examples["ner_tags"],
        truncation=True,
        padding="max_length",
    )

# small slice for a throughput probe, not a real fine-tune
probe = dataset["train"].select(range(32))
encoded = probe.map(prepare, batched=True, remove_columns=probe.column_names)
encoded.set_format("torch")


In [ ]:
import time
from torch.utils.data import DataLoader

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=len(label_list)
).to("cuda")
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

loader = DataLoader(encoded, batch_size=4, shuffle=True)

model.train()
step_times = []
for i, batch in enumerate(loader):
    if i >= 20:
        break
    batch = {k: v.to("cuda") for k, v in batch.items()}
    torch.cuda.synchronize()
    t0 = time.time()

    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    torch.cuda.synchronize()
    step_times.append(time.time() - t0)

warm = step_times[2:]  # drop first couple steps for warmup
avg = sum(warm) / len(warm)
print(f"steps timed: {len(warm)}")
print(f"avg step time: {avg:.3f}s  ({1/avg:.2f} steps/sec)")
if avg > 2.0:
    print("WARNING: >2s/step on a T4/P100 at batch=4 is much slower than expected — "
          "re-check the transformers version and fp16/precision settings before real training.")
else:
    print("Looks healthy — proceed to real fine-tuning.")

## Checklist
- [ ] GPU detected (`torch.cuda.is_available()` True)
- [ ] transformers version printed (informational — not required to match any specific pin)
- [ ] Avg step time printed above and looks sane (no order-of-magnitude blowup) — this is the real regression check
- [ ] Record the avg step time, GPU type, and transformers version for reference before starting real training
